 # 01 · Ingesta Bronze — Orquestador YelpConnect

 **Tipo:** Orquestador de etapa  
 **Capa:** Bronze  
 **Modo:** Secuencial · Fail-Fast  
 **Punto de entrada del Job:** Sí

 ---

 ## Propósito

 Exponer un único punto de entrada para ejecutar RAW → Bronze:

 ```text
 01_ingestar_bronze
        │
        ├── _bronze/01_ingestar_batch
        │       ├── Business
        │       ├── User
        │       ├── Tip
        │       └── Check-in
        │
        ├── _bronze/02_ingestar_review
        │       └── Auto Loader + checkpoint
        │
        └── _bronze/03_validar_bronze
                └── Quality Gate técnico
 ```

 El orquestador no contiene lógica de transformación.
 Solo ordena, ejecuta, detiene ante fallos y registra resultados.

In [0]:
# COMMAND ----------

from datetime import datetime, timezone
import json

STEPS = [
    {
        "order": 1,
        "name": "Ingesta Bronze Batch",
        "path": "./_bronze/01_ingestar_batch",
        "timeout_seconds": 1800
    },
    {
        "order": 2,
        "name": "Ingesta Bronze Review",
        "path": "./_bronze/02_ingestar_review",
        "timeout_seconds": 1800
    },
    {
        "order": 3,
        "name": "Validación Técnica Bronze",
        "path": "./_bronze/03_validar_bronze",
        "timeout_seconds": 1200
    }
]

def utc_now():
    return datetime.now(timezone.utc)

# COMMAND ----------

 ## Estrategia Fail-Fast

 - Si falla Batch, Review no se ejecuta.
 - Si falla Review, el Quality Gate no se ejecuta.
 - Si falla el Quality Gate, Bronze se considera fallido.

 El siguiente bloque del pipeline solo puede iniciar si este notebook
 termina correctamente.

In [0]:
# COMMAND ----------

results = []

for step in STEPS:

    started_at = utc_now()

    print()
    print("=" * 78)
    print(
        f"PASO {step['order']}/{len(STEPS)} · {step['name']}"
    )
    print(f"Notebook: {step['path']}")
    print(f"Inicio UTC: {started_at.isoformat()}")
    print("=" * 78)

    try:

        child_output = dbutils.notebook.run(
            step["path"],
            step["timeout_seconds"],
            {}
        )

        finished_at = utc_now()
        duration = (
            finished_at - started_at
        ).total_seconds()

        try:
            parsed_output = (
                json.loads(child_output)
                if child_output
                else {}
            )
        except Exception:
            parsed_output = {
                "raw_output": child_output
            }

        results.append({
            "order": step["order"],
            "step": step["name"],
            "notebook": step["path"],
            "status": "SUCCESS",
            "started_at_utc": started_at.isoformat(),
            "finished_at_utc": finished_at.isoformat(),
            "duration_seconds": round(duration, 2),
            "detail": json.dumps(parsed_output)
        })

        print(
            f"✔ {step['name']} ({duration:.2f}s)"
        )

    except Exception as exc:

        finished_at = utc_now()
        duration = (
            finished_at - started_at
        ).total_seconds()

        results.append({
            "order": step["order"],
            "step": step["name"],
            "notebook": step["path"],
            "status": "FAILED",
            "started_at_utc": started_at.isoformat(),
            "finished_at_utc": finished_at.isoformat(),
            "duration_seconds": round(duration, 2),
            "detail": str(exc)[:4000]
        })

        display(
            spark.createDataFrame(results)
            .orderBy("order")
        )

        raise RuntimeError(
            "La etapa Bronze se detuvo en "
            f"{step['name']}: {exc}"
        ) from exc


In [0]:
# COMMAND ----------

results_df = (
    spark.createDataFrame(results)
    .orderBy("order")
)

display(results_df)

successful_steps = (
    results_df
    .filter("status = 'SUCCESS'")
    .count()
)

if successful_steps != len(STEPS):
    raise RuntimeError(
        "La etapa Bronze no terminó completamente."
    )

# COMMAND ----------

 ## Definition of Done

 `01_ingestar_bronze` se considera exitoso cuando:

 - [ ] Batch finaliza correctamente;
 - [ ] Review Auto Loader finaliza correctamente;
 - [ ] las cinco tablas Bronze existen;
 - [ ] todas poseen metadata técnica;
 - [ ] el Quality Gate técnico retorna PASS;
 - [ ] no se aplicó lógica de negocio.

 ---

 ## Siguiente etapa

 `E5/02_evaluar_dq_bronze`

 antes de permitir la transformación a Silver.

In [0]:
# COMMAND ----------

total_duration = sum(
    row["duration_seconds"]
    for row in results
)

final_summary = {
    "status": "SUCCESS",
    "stage": "bronze",
    "steps": successful_steps,
    "duration_seconds": round(total_duration, 2)
}

print()
print("=" * 78)
print("ETAPA BRONZE COMPLETADA")
print("=" * 78)
print(
    f"Pasos exitosos : {successful_steps}/{len(STEPS)}"
)
print(
    f"Duración total : {total_duration:.2f} segundos"
)
print("Quality Gate   : PASS")
print("=" * 78)

dbutils.notebook.exit(
    json.dumps(final_summary)
)
